In [ ]:
!pip install matplotlib


In [ ]:
!pip install TotalSegmentator


In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

# Paths
ct_path = r"C:\Users\paula\Downloads\SAROS_CT_IMAGES\case_000\image.nii.gz"
seg_path = r"SAROS-Collection-NIfTI-files-v2_03-70-2024/SAROS-Collection-NIfTI-files-v2_03-70-2024/case_000/body-regions.nii.gz"

# Load NIfTI files
ct_nii = nib.load(ct_path)
seg_nii = nib.load(seg_path)

# Get image arrays
ct = ct_nii.get_fdata()
seg = seg_nii.get_fdata()

print("CT shape:", ct.shape)
print("Segmentation shape:", seg.shape)

print("CT affine:\n", ct_nii.affine)
print("Seg affine:\n", seg_nii.affine)

In [ ]:
# Select slice
slice_idx = ct.shape[2] // 3
seg_filtered = seg.where(seg == slice_idx)
plt.figure(figsize=(8, 8))

# CT
plt.imshow(
    ct[:, :, slice_idx].T,
    cmap="gray",
    origin="lower"
)

# Segmentation overlay
plt.imshow(
    np.ma.masked_where(seg[:, :, slice_idx].T == 0,
                       seg[:, :, slice_idx].T),
    cmap="jet",
    alpha=0.5,
    origin="lower",
    interpolation="nearest"
)

plt.title(f"Axial slice {slice_idx}")
plt.axis("off")
plt.show()

In [ ]:
# Display every 10th slice
step = 10

for slice_idx in range(0, ct.shape[2], step):

    fig, ax = plt.subplots(figsize=(7, 7))

    ax.imshow(
        ct[:, :, slice_idx].T,
        cmap="gray",
        origin="lower"
    )

    ax.imshow(
        np.ma.masked_where(
            seg[:, :, slice_idx].T == 0,
            seg[:, :, slice_idx].T
        ),
        cmap="jet",
        alpha=0.5,
        origin="lower",
        interpolation="nearest"
    )

    ax.set_title(f"Slice {slice_idx}")
    ax.axis("off")

    plt.show()

In [ ]:
labels, counts = np.unique(seg, return_counts=True)

for label, count in zip(labels, counts):
    print(f"Label {label}: {count} voxels")

In [ ]:
import matplotlib.patches as mpatches

# Get unique segmentation labels, excluding background (0)
# labels = np.unique(seg).astype(int)
# labels = labels[labels != 0]

label_mapping = {
0: "Background",
1: "Subcutaneous Tissue",
2: "Muscle",
3: "Abdominal Cavity",
4:"Thoracic Cavity",
5:"Bones",
6: "Parotid Glands",
7:"Pericardium",
8:"Breast Implant",
9:"Mediastinum",
10:"Brain",
11:"Spinal Cord",
12:"Thyroid Glands",
13:"Submandibular Glands"
}
# Keep only labels defined in label_mapping
valid_labels = list(label_mapping.keys())

# Set all other labels to background (0)
seg[~np.isin(seg, valid_labels)] = 0
# Only include labels actually present, excluding background
labels = np.unique(seg).astype(int)
labels = labels[labels != 0]

# Create a consistent colormap for all slices
cmap = plt.get_cmap("tab10", len(labels))

# Map each label to a colour
label_colors = {
    label: cmap(i) for i, label in enumerate(labels)
}

# Display every 10th slice
step = 5

for slice_idx in range(0, ct.shape[2], step):

    fig, ax = plt.subplots(figsize=(9, 8))

    ax.imshow(
        ct[:, :, slice_idx].T,
        cmap="gray",
        origin="lower"
    )

    # Overlay each label separately
    for label, color in label_colors.items():
        mask = np.ma.masked_where(
            seg[:, :, slice_idx].T != label,
            seg[:, :, slice_idx].T
        )

        ax.imshow(
            mask,
            cmap=plt.matplotlib.colors.ListedColormap([color]),
            alpha=0.5,
            origin="lower",
            interpolation="nearest"
        )

      # Named legend
    legend_handles = [
        mpatches.Patch(
            color=label_colors[label],
            label=f"{label}: {label_mapping[label]}"
        )
        for label in labels
    ]

    ax.legend(
        handles=legend_handles,
        loc="upper left",
        bbox_to_anchor=(1.02, 1),
        title="Segmentation labels"
    )

    ax.set_title(f"Axial slice {slice_idx}")
    ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
from pathlib import Path
folder = Path("segmentator_output/case_002")
%matplotlib inline
for file in folder.iterdir():
    vertebra = file.name.replace("vertebrae_", "").replace(".nii.gz", "")

    # Load segmentation
    seg_nii = nib.load(str(file))
    seg = seg_nii.get_fdata()
        # Select slice

    # Find all axial slices containing this vertebra
    slice_indices = np.where(np.any(seg > 0, axis=(0, 1)))[0]

    print(f"{vertebra}: {len(slice_indices)} slices")

    for slice_idx in slice_indices:

        plt.figure(figsize=(8, 8))

        # CT
        plt.imshow(
            ct[:, :, slice_idx].T,
            cmap="gray",
            origin="lower"
        )

        # Segmentation overlay
        plt.imshow(
            np.ma.masked_where(
                seg[:, :, slice_idx].T == 0,
                seg[:, :, slice_idx].T
            ),
            cmap="autumn",
            alpha=0.5,
            origin="lower",
            interpolation="nearest"
        )

        plt.title(f"{vertebra} - Axial slice {slice_idx}")
        plt.axis("off")
        plt.show()

In [ ]:
# Get voxel spacing in mm
spacing = ct_nii.header.get_zooms()

pixel_area_mm2 = spacing[0] * spacing[1]

print("Pixel area:", pixel_area_mm2, "mm²")

In [ ]:
seg_path = r"SAROS-Collection-NIfTI-files-v2_03-70-2024/SAROS-Collection-NIfTI-files-v2_03-70-2024/case_002/body-regions.nii.gz"

# Load NIfTI files
ct_nii = nib.load(ct_path)
seg_nii_comp = nib.load(seg_path)

# Get image arrays
ct = ct_nii.get_fdata()
seg_comp = seg_nii_comp.get_fdata()

In [ ]:
print("Shape:", seg_comp.shape)
print("Unique labels:", np.unique(seg_comp))
print("Nonzero voxels:", np.sum(seg_comp > 0))
print("Voxel spacing:", seg_nii_comp.header.get_zooms())

In [ ]:
# Find slices containing at least one annotated voxel
annotated_slices = np.where(
    np.any(np.isin(seg_comp, [1, 2]), axis=(0, 1))
)[0]

print("Annotated slices:", annotated_slices)

In [ ]:
### find center slice of each vertebra
import pandas as pd
results = []
annotated_slices = np.where(
    np.any(np.isin(seg_comp, [1, 2]), axis=(0, 1))
)[0]

for file in sorted(folder.glob("vertebrae_*.nii.gz")):

    # Extract vertebra name
    vertebra = file.name.replace("vertebrae_", "").replace(".nii.gz", "")

    # Load vertebra segmentation
    vertebra_nii = nib.load(str(file))
    vertebra_seg = vertebra_nii.get_fdata()

    # Find slice with the largest vertebral cross-section
    slice_areas = np.sum(vertebra_seg > 0, axis=(0, 1))

     # Skip empty segmentations
    if np.max(slice_areas) == 0:
        print(f"{vertebra}: No segmentation found. Skipping.")
        continue

    max_area = np.max(slice_areas)

    # All slices with the maximum number of pixels
    max_slices = np.where(slice_areas == max_area)[0]

    # Take median slice
    slice_idx = int(max_slices[len(max_slices) // 2])
    #slice_idx = np.argmax(slice_areas)


    # Find nearest annotated body-composition slice
    nearest_annotated = annotated_slices[
    np.argmin(np.abs(annotated_slices - slice_idx))
    ]
    # Calculate body composition areas at this slice
    body_slice = seg_comp[:, :, nearest_annotated]


    print(f"{vertebra}: vertebra slice {slice_idx}, "
      f"body composition slice {nearest_annotated}")
    slice_difference = abs(nearest_annotated - slice_idx)
    distance_mm = slice_difference * spacing[2]

    n_fat = np.sum(body_slice == 1)
    n_muscle = np.sum(body_slice == 2)


    # Convert pixel count to cm²
    area_subcutaneous = n_fat * pixel_area_mm2 / 100
    area_muscle = n_muscle * pixel_area_mm2 / 100

    results.append({
        "Vertebra": vertebra,
        "Slice": slice_idx,
        "Selected slice": nearest_annotated,
        "Slice difference": slice_difference,
        "Distance (mm)": distance_mm,
        "Subcutaneous Tissue (mm²)": area_subcutaneous,
        "Muscle (mm²)": area_muscle
    })

df_landmarks = pd.DataFrame(results)

print(df_landmarks)

In [ ]:
df_landmarks.columns

In [ ]:
import pandas as pd
results = []

# Every 5th slice is annotated
for slice_idx in range(0, seg_comp.shape[2], 5):

    slice_seg = seg_comp[:, :, slice_idx]

    # Count pixels belonging to each label
    n_subcutaneous = np.sum(slice_seg == 1)
    n_muscle = np.sum(slice_seg == 2)

    # Convert pixel count to cm²
    area_subcutaneous = n_subcutaneous * pixel_area_mm2 / 100
    area_muscle = n_muscle * pixel_area_mm2 / 100

    results.append({
        "slice": slice_idx,
        "Subcutaneous Tissue (cm2)": area_subcutaneous,
        "Muscle (cm2)": area_muscle
    })

# Convert to DataFrame
df = pd.DataFrame(results)

print(df.head())

In [ ]:
# Map selected body-composition slice to vertebral level
vertebra_mapping = dict(zip(
    df_landmarks["Selected slice"],
    df_landmarks["Vertebra"]
))

# Add vertebra column
df["Vertebra"] = df["slice"].map(vertebra_mapping)

In [ ]:
from totalsegmentator.python_api import totalsegmentator
from pathlib import Path

ct_path = Path( r"C:\Users\paula\Downloads\SAROS_CT_IMAGES\case_002\image.nii.gz")
output_dir = Path("segmentator_output/case_002")

output_dir.mkdir(parents=True, exist_ok=True)

vertebrae = [
    f"vertebrae_{level}"
    for level in [
        "T1", "T2", "T3", "T4", "T5", "T6",
        "T7", "T8", "T9", "T10", "T11", "T12",
        "L1", "L2", "L3", "L4", "L5"
    ]
]

totalsegmentator(
    input=str(ct_path),
    output=str(output_dir),
    task="total",
    roi_subset=vertebrae,
    device="cpu",
    fast=True
)

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    df["slice"],
    df["Subcutaneous Tissue (cm2)"],
    marker="o",
    label="Subcutaneous Tissue"
)

plt.plot(
    df["slice"],
    df["Muscle (cm2)"],
    marker="o",
    label="Muscle"
)

plt.xlabel("Slice index")
plt.ylabel("Cross-sectional area (cm²)")
plt.title("Body composition along the torso")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:


fig, ax = plt.subplots(figsize=(13, 5))

x = np.arange(len(df))

# Plot tissue areas
ax.plot(
    x,
    df["Subcutaneous Tissue (cm2)"],
    marker="o",
    label="Subcutaneous Tissue"
)

ax.plot(
    x,
    df["Muscle (cm2)"],
    marker="o",
    label="Muscle"
)

# Bottom axis: vertebral levels
ax.set_xticks(x)
ax.set_xticklabels(df["slice"])

ax.set_xlabel("Slice")
ax.set_ylabel("Cross-sectional area (mm²)")
ax.set_title("Body composition along the torso")

# Top axis: slice indices
ax_top = ax.secondary_xaxis("top")
ax_top.set_xticks(x)
ax_top.set_xticklabels(
    df["Vertebra"].fillna("")
)
ax_top.set_xlabel("CT slice index")

ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Sort by selected slice index
df_landmarks = df_landmarks.sort_values("Selected slice").reset_index(drop=True)


fig, ax = plt.subplots(figsize=(13, 5))

x = np.arange(len(df_landmarks))

# Plot tissue areas
ax.plot(
    x,
    df_landmarks["Subcutaneous Tissue (mm²)"],
    marker="o",
    label="Subcutaneous Tissue"
)

ax.plot(
    x,
    df_landmarks["Muscle (mm²)"],
    marker="o",
    label="Muscle"
)

# Bottom axis: vertebral levels
ax.set_xticks(x)
ax.set_xticklabels(df_landmarks["Vertebra"])

ax.set_xlabel("Vertebral level")
ax.set_ylabel("Cross-sectional area (mm²)")
ax.set_title("Body composition along the torso")

# Top axis: slice indices
ax_top = ax.secondary_xaxis("top")
ax_top.set_xticks(x)
ax_top.set_xticklabels(df_landmarks["Selected slice"])
ax_top.set_xlabel("CT slice index")

ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()